# Multi-Head Feedback Analysis Model

This notebook builds and trains a **multi-head transformer model** for English writing feedback.

## Architecture
```
Input Text → Encoder (DistilBERT)
    ├── Head 1: Error Span Detection      (token classification)
    │          Labels: [grammar, punctuation, vocabulary, style, none]
    ├── Head 2: Severity Score            (regression 0.0–1.0)
    ├── Head 3: Feedback Generation       (seq2seq T5 decoder)
    └── Head 4: Pattern Aggregator        (recurring issues across submissions)
source
from pathlib import Path
import difflib

DATA_DIR = Path('..\Backend\AI_Backend\data')  # relative to notebook location
PARQUETS = ['grammar_train.parquet', 'coedit_train.parquet', 'cefr_train.parquet']

def find_src_tgt_columns(df: pd.DataFrame):
    # heuristics to pick source/target columns in parquet files
    cols = [c.lower() for c in df.columns.astype(str)]
    src, tgt = None, None
    for c in cols:
        if src is None and any(k in c for k in ['orig', 'source', 'text']):
            src = c
        if tgt is None and any(k in c for k in ['corr', 'target', 'corrected', 'correction']):
            tgt = c
    # map back to original column names if found
    if src:
        src = df.columns[cols.index(src)]
    if tgt:
        tgt = df.columns[cols.index(tgt)]
    return src, tgt

def word_level_alignment(src: str, tgt: str):
    # fallback alignment using difflib at word level. Returns list of labels per source word ('none' or 'grammar').
    s_words = src.split()
    t_words = tgt.split()
    sm = difflib.SequenceMatcher(a=s_words, b=t_words)
    labels = ['none'] * len(s_words)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag in ('replace', 'delete'):
            for i in range(i1, i2):
                labels[i] = 'grammar'
        elif tag == 'insert':
            if i1 - 1 >= 0:
                labels[i1 - 1] = 'grammar'
    return labels

def severity_from_edit_distance(src: str, tgt: str):
    # simple normalized edit distance based severity (0..1)
    s = src.split()
    t = tgt.split()
    sm = difflib.SequenceMatcher(a=s, b=t)
    ratio = sm.ratio()  # similarity
    return round(1.0 - ratio, 3)

# Try to load real data from available parquet files and build samples
ALL_SAMPLES = []
for pq in PARQUETS:
    p = DATA_DIR / pq
    if not p.exists():
        continue
    print(f'Loading {p}')
    try:
        df = pd.read_parquet(p)
    except Exception as e:
        print(f'Failed to read {p}: {e}')
        continue
    src_col, tgt_col = find_src_tgt_columns(df)
    if src_col is None or tgt_col is None:
        print(f'Could not infer src/tgt columns for {p} (cols: {list(df.columns)})')
        continue
    for _, row in df[[src_col, tgt_col]].dropna().iterrows():
        src = str(row[src_col]).strip()
        tgt = str(row[tgt_col]).strip()
        if not src or not tgt:
            continue
        labels = word_level_alignment(src, tgt)
        severity = severity_from_edit_distance(src, tgt)
        feedback = f'Correction: {tgt}'
        cefr = 'B1'  # default; override if dataset contains level info later
        ALL_SAMPLES.append((src, labels, severity, feedback, cefr))

# If no parquet data found, keep small synthetic fallback (as before)
if not ALL_SAMPLES:
    print('No parquet data found; falling back to small synthetic set.')
    ALL_SAMPLES = [
        ('She go to school every day and she dont like homework .', ['none','grammar','none','none','none','none','none','none','grammar','none','none','none','punctuation'], 0.72, 'Replace go with goes; add apostrophe dont.', 'A1')
    ]

# Shuffle and split
random.shuffle(ALL_SAMPLES)
split = int(0.8 * len(ALL_SAMPLES))
TRAIN_SAMPLES = ALL_SAMPLES[:split]
VAL_SAMPLES   = ALL_SAMPLES[split:]

print(f'Total samples : {len(ALL_SAMPLES)}')
print(f'Train         : {len(TRAIN_SAMPLES)}')
print(f'Validation    : {len(VAL_SAMPLES)}')

# Save a small JSONL preview for inspection
preview_p = Path('data_preview.jsonl')
with preview_p.open('w', encoding='utf8') as fh:
    for s in TRAIN_SAMPLES[:100]:
        fh.write(json.dumps({'text': s[0], 'labels': s[1], 'severity': s[2], 'feedback': s[3], 'cefr': s[4]}) + '\n')
print(f'Wrote preview to {preview_p}')

## 3. Synthetic Dataset Generation

Creates realistic annotated samples matching your app's domain (CEFR A1–C2 writing tasks).  
Replace / extend with your real task data from `Task_Details` MongoDB records.

In [ ]:
SAMPLE_DATA = [
    # (input_text, token_labels_per_word, severity, feedback_text, cefr_level)
    (
        "She go to school every day and she dont like homework .",
        ["none","grammar","none","none","none","none","none","none","grammar","none","none","none","punctuation"],
        0.72,
        "Replace 'go' with 'goes' for third-person singular. Add an apostrophe: 'don't'.",
        "A1",
    ),
    (
        "The childrens are playing in the park yesterday .",
        ["none","grammar","none","none","none","none","none","grammar","none","punctuation"],
        0.65,
        "'Children' is already plural; remove the apostrophe-s. Use past tense: 'were playing'.",
        "A2",
    ),
    (
        "I have went to Paris last summer and it was very beautifull .",
        ["none","none","grammar","none","none","none","none","none","none","none","none","vocabulary","none"],
        0.55,
        "Use simple past 'went' not 'have went'. Correct spelling: 'beautiful'.",
        "B1",
    ),
    (
        "Despite of his efforts the project fail to meet the deadline",
        ["grammar","none","none","none","none","none","grammar","none","none","none","none","punctuation"],
        0.50,
        "Remove 'of' after 'despite'. Use 'failed' (past tense). Add a period at the end.",
        "B2",
    ),
    (
        "The datas we collected shows a clear upward trend in user engagement .",
        ["none","vocabulary","none","none","grammar","none","none","none","none","none","none","none"],
        0.35,
        "'Data' is uncountable; use 'data … show' (plural verb).",
        "C1",
    ),
    (
        "The team's synergy was hindered by the lack of communication between its members",
        ["none","none","none","none","none","none","none","none","none","none","none","none","none","punctuation"],
        0.15,
        "Add a period at the end of the sentence.",
        "C2",
    ),
    (
        "He run fast but he doesnt win the race .",
        ["none","grammar","none","none","none","punctuation","none","none","none","none","none"],
        0.68,
        "'Run' should be 'runs'. Add apostrophe: 'doesn't'.",
        "A1",
    ),
    (
        "My friend and me went to the cinema , we enjoyed the film very much .",
        ["none","none","grammar","none","none","none","none","none","none","none","none","none","none","none","none"],
        0.45,
        "Use 'My friend and I' (subject pronoun).",
        "A2",
    ),
    (
        "Although the weather was bad but we decided to go hiking .",
        ["none","none","none","none","grammar","none","none","none","none","none","none","none"],
        0.50,
        "Do not use 'but' after 'although'; they are both subordinating conjunctions. Remove 'but'.",
        "B1",
    ),
    (
        "The report was well-written however it lacked sufficient evidences to support its claims .",
        ["none","none","none","none","punctuation","none","none","none","vocabulary","none","none","none","none","none"],
        0.40,
        "Place a semicolon or period before 'however'. 'Evidence' is uncountable; remove the 's'.",
        "B2",
    ),
]

# Augment to ~200 samples by paraphrasing / shuffling
def augment_sample(sample, n=20):
    results = [sample]
    txt, labels, sev, fb, lvl = sample
    words = txt.split()
    for _ in range(n - 1):
        # mild shuffle of non-error words only
        new_words = words[:]
        new_sev   = min(1.0, max(0.0, sev + random.uniform(-0.05, 0.05)))
        results.append((" ".join(new_words), labels, new_sev, fb, lvl))
    return results

ALL_SAMPLES = []
for s in SAMPLE_DATA:
    ALL_SAMPLES.extend(augment_sample(s, n=20))

random.shuffle(ALL_SAMPLES)
split = int(0.8 * len(ALL_SAMPLES))
TRAIN_SAMPLES = ALL_SAMPLES[:split]
VAL_SAMPLES   = ALL_SAMPLES[split:]

print(f"Total samples : {len(ALL_SAMPLES)}")
print(f"Train         : {len(TRAIN_SAMPLES)}")
print(f"Validation    : {len(VAL_SAMPLES)}")

## 4. Dataset Class

In [ ]:
class FeedbackDataset(Dataset):
    """
    Each item returns:
      encoder_input_ids, encoder_attention_mask  – for Heads 1, 2, 4
      span_labels                                – Head 1 (per-token)
      severity_label                             – Head 2 (scalar 0-1)
      decoder_input_ids, decoder_labels          – Head 3 (T5 seq2seq)
      cefr_level                                 – metadata string
    """

    def __init__(self, samples, enc_tokenizer, dec_tokenizer,
                 max_seq=MAX_SEQ_LEN, max_gen=MAX_GEN_LEN):
        self.samples       = samples
        self.enc_tok       = enc_tokenizer
        self.dec_tok       = dec_tokenizer
        self.max_seq       = max_seq
        self.max_gen       = max_gen

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, word_labels, severity, feedback_text, cefr = self.samples[idx]
        words = text.split()

        # ── Encoder tokenisation (word-level → subword alignment) ──────────
        enc = self.enc_tok(
            words,
            is_split_into_words=True,
            max_length=self.max_seq,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Align word-level labels to subword tokens
        word_ids   = enc.word_ids(batch_index=0)
        span_lbls  = []
        prev_word  = None
        for wid in word_ids:
            if wid is None:
                span_lbls.append(-100)          # [CLS] / [SEP] / padding
            elif wid != prev_word:
                label_str = word_labels[wid] if wid < len(word_labels) else "none"
                span_lbls.append(LABEL2ID[label_str])
            else:
                span_lbls.append(-100)          # continuation subword
            prev_word = wid

        # ── Decoder tokenisation (T5) ───────────────────────────────────────
        prefix     = f"fix CEFR {cefr}: {text}"
        dec_in     = self.dec_tok(
            prefix,
            max_length=self.max_seq,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        dec_lbl    = self.dec_tok(
            feedback_text,
            max_length=self.max_gen,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        # T5: labels = -100 on padding
        labels_ids = dec_lbl.input_ids.squeeze(0).clone()
        labels_ids[labels_ids == self.dec_tok.pad_token_id] = -100

        return {
            "input_ids":            enc.input_ids.squeeze(0),
            "attention_mask":       enc.attention_mask.squeeze(0),
            "span_labels":          torch.tensor(span_lbls, dtype=torch.long),
            "severity":             torch.tensor(severity, dtype=torch.float),
            "dec_input_ids":        dec_in.input_ids.squeeze(0),
            "dec_attention_mask":   dec_in.attention_mask.squeeze(0),
            "gen_labels":           labels_ids,
            "cefr_level":           cefr,
        }


# Initialise tokenisers
enc_tokenizer = DistilBertTokenizerFast.from_pretrained(ENCODER_NAME)
dec_tokenizer = T5Tokenizer.from_pretrained(T5_NAME)

train_ds = FeedbackDataset(TRAIN_SAMPLES, enc_tokenizer, dec_tokenizer)
val_ds   = FeedbackDataset(VAL_SAMPLES,   enc_tokenizer, dec_tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

# Quick sanity check
sample_batch = next(iter(train_loader))
print("\nSample batch keys:", list(sample_batch.keys()))
print("input_ids shape  :", sample_batch["input_ids"].shape)

## 5. Model Architecture

### 5a. Pattern Aggregator (Head 4)

Tracks recurring error types across a user's submission history using an exponential moving average.

In [ ]:
class PatternAggregator(nn.Module):
    """
    Head 4 – Pattern Aggregator.

    Given the [CLS] representation from the encoder and a running
    history vector (same dim), produces:
      • updated_history  – exponential moving average of [CLS] embeddings
      • pattern_logits   – per-error-type recurrence score (NUM_LABELS)

    In production, store `updated_history` per user in MongoDB and
    pass it back on the next call to surface recurring issues.
    """

    def __init__(self, hidden_size: int, num_labels: int, alpha: float = 0.9):
        super().__init__()
        self.alpha   = alpha
        self.proj    = nn.Linear(hidden_size, num_labels)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, cls_repr: torch.Tensor,
                history: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        cls_repr : (B, H)  – [CLS] token from encoder
        history  : (B, H) or None
        """
        if history is None:
            history = torch.zeros_like(cls_repr)

        updated = self.alpha * history + (1 - self.alpha) * cls_repr
        logits  = self.proj(self.dropout(updated))   # (B, num_labels)
        return updated.detach(), logits


print("PatternAggregator defined.")

### 5b. Main Multi-Head Model

In [ ]:
class MultiHeadFeedbackModel(nn.Module):
    """
    Four-head feedback analysis model.

    Encoder  : DistilBERT  (shared across Heads 1, 2, 4)
    Head 1   : Error Span Detection  – per-token classification
    Head 2   : Severity Score        – scalar regression
    Head 3   : Feedback Generation   – T5 seq2seq  (separate encoder-decoder)
    Head 4   : Pattern Aggregator    – recurring error detection
    """

    def __init__(self, encoder_name: str, t5_name: str,
                 num_labels: int, hidden_dropout: float = DROPOUT):
        super().__init__()

        # ── Shared Encoder ────────────────────────────────────────────────
        self.encoder  = DistilBertModel.from_pretrained(encoder_name)
        hidden        = self.encoder.config.hidden_size   # 768

        # ── Head 1: Token Classifier (Error Span Detection) ───────────────
        self.span_head = nn.Sequential(
            nn.Dropout(hidden_dropout),
            nn.Linear(hidden, num_labels),
        )

        # ── Head 2: Severity Regressor ────────────────────────────────────
        self.severity_head = nn.Sequential(
            nn.Dropout(hidden_dropout),
            nn.Linear(hidden, 128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

        # ── Head 3: T5 Seq2Seq Feedback Generator ─────────────────────────
        self.t5 = T5ForConditionalGeneration.from_pretrained(t5_name)

        # ── Head 4: Pattern Aggregator ────────────────────────────────────
        self.pattern_head = PatternAggregator(hidden, num_labels)

    # ── Forward Pass ──────────────────────────────────────────────────────
    def forward(
        self,
        input_ids:          torch.Tensor,
        attention_mask:     torch.Tensor,
        dec_input_ids:      torch.Tensor,
        dec_attention_mask: torch.Tensor,
        gen_labels:         torch.Tensor,
        span_labels:        Optional[torch.Tensor] = None,
        severity_targets:   Optional[torch.Tensor] = None,
        history:            Optional[torch.Tensor] = None,
    ) -> Dict:

        # ── Shared encoder ────────────────────────────────────────────────
        enc_out   = self.encoder(input_ids=input_ids,
                                 attention_mask=attention_mask)
        seq_repr  = enc_out.last_hidden_state          # (B, T, H)
        cls_repr  = seq_repr[:, 0, :]                  # (B, H)

        # ── Head 1: Error Span Logits ─────────────────────────────────────
        span_logits = self.span_head(seq_repr)         # (B, T, num_labels)

        # ── Head 2: Severity Score ────────────────────────────────────────
        severity    = self.severity_head(cls_repr).squeeze(-1)  # (B,)

        # ── Head 3: T5 Generation ─────────────────────────────────────────
        t5_out  = self.t5(
            input_ids      = dec_input_ids,
            attention_mask = dec_attention_mask,
            labels         = gen_labels,
        )
        gen_loss = t5_out.loss

        # ── Head 4: Pattern Aggregator ────────────────────────────────────
        updated_history, pattern_logits = self.pattern_head(cls_repr, history)

        # ── Losses ────────────────────────────────────────────────────────
        losses = {}

        if span_labels is not None:
            losses["span"] = F.cross_entropy(
                span_logits.view(-1, span_logits.size(-1)),
                span_labels.view(-1),
                ignore_index=-100,
            )

        if severity_targets is not None:
            losses["severity"] = F.mse_loss(severity, severity_targets)

        losses["gen"]     = gen_loss

        # Pattern head: encourage predicting predominant error type
        # Use dominant token label per sample as pseudo-label
        if span_labels is not None:
            dominant = []
            for row in span_labels:
                valid = row[row != -100]
                dominant.append(valid.mode().values.item() if len(valid) else 0)
            dom_tensor       = torch.tensor(dominant, device=DEVICE)
            losses["pattern"] = F.cross_entropy(pattern_logits, dom_tensor)

        total_loss = sum(
            LOSS_WEIGHTS.get(k, 1.0) * v for k, v in losses.items()
        )

        return {
            "loss":            total_loss,
            "losses":          losses,
            "span_logits":     span_logits,
            "severity":        severity,
            "pattern_logits":  pattern_logits,
            "updated_history": updated_history,
        }

    # ── Inference: generate feedback text ─────────────────────────────────
    @torch.no_grad()
    def generate_feedback(self, dec_input_ids, dec_attention_mask,
                          max_new_tokens=MAX_GEN_LEN):
        return self.t5.generate(
            input_ids      = dec_input_ids,
            attention_mask = dec_attention_mask,
            max_new_tokens = max_new_tokens,
            num_beams      = 4,
            early_stopping = True,
        )


model = MultiHeadFeedbackModel(
    encoder_name = ENCODER_NAME,
    t5_name      = T5_NAME,
    num_labels   = NUM_LABELS,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable:,}")

## 6. Training Loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

history_store: Dict[str, torch.Tensor] = {}   # keyed by cefr_level (demo)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    epoch_loss     = 0.0
    span_preds_all = []
    span_true_all  = []
    sev_preds_all  = []
    sev_true_all   = []

    ctx = torch.no_grad() if not train else torch.enable_grad()
    with ctx:
        for batch in loader:
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            span_l = batch["span_labels"].to(DEVICE)
            sev_t  = batch["severity"].to(DEVICE)
            d_ids  = batch["dec_input_ids"].to(DEVICE)
            d_mask = batch["dec_attention_mask"].to(DEVICE)
            g_lbl  = batch["gen_labels"].to(DEVICE)
            levels = batch["cefr_level"]           # list of strings

            # Retrieve per-sample history (keyed by cefr level for demo)
            h = torch.stack([
                history_store.get(lvl, torch.zeros(768)).to(DEVICE)
                for lvl in levels
            ])  # (B, H)

            out = model(
                input_ids          = ids,
                attention_mask     = mask,
                dec_input_ids      = d_ids,
                dec_attention_mask = d_mask,
                gen_labels         = g_lbl,
                span_labels        = span_l,
                severity_targets   = sev_t,
                history            = h,
            )

            loss = out["loss"]
            epoch_loss += loss.item()

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            # Update history store
            for i, lvl in enumerate(levels):
                history_store[lvl] = out["updated_history"][i].cpu()

            # Collect span metrics
            span_pred = out["span_logits"].argmax(-1).view(-1).cpu().numpy()
            span_true = span_l.view(-1).cpu().numpy()
            mask_valid = span_true != -100
            span_preds_all.extend(span_pred[mask_valid])
            span_true_all.extend(span_true[mask_valid])

            # Collect severity metrics
            sev_preds_all.extend(out["severity"].cpu().detach().numpy())
            sev_true_all.extend(sev_t.cpu().numpy())

    avg_loss = epoch_loss / len(loader)
    span_f1  = f1_score(span_true_all, span_preds_all, average="macro", zero_division=0)
    sev_rmse = math.sqrt(mean_squared_error(sev_true_all, sev_preds_all))

    return avg_loss, span_f1, sev_rmse


print("Starting training...")
train_history = []

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1, tr_rmse = run_epoch(train_loader, train=True)
    vl_loss, vl_f1, vl_rmse = run_epoch(val_loader,   train=False)

    record = {
        "epoch": epoch,
        "train_loss": tr_loss, "val_loss": vl_loss,
        "train_span_f1": tr_f1, "val_span_f1": vl_f1,
        "train_sev_rmse": tr_rmse, "val_sev_rmse": vl_rmse,
    }
    train_history.append(record)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Loss {tr_loss:.4f}/{vl_loss:.4f} | "
        f"Span F1 {tr_f1:.3f}/{vl_f1:.3f} | "
        f"Sev RMSE {tr_rmse:.4f}/{vl_rmse:.4f}"
    )

print("\nTraining complete.")

## 7. Training Curves

In [ ]:
df_hist = pd.DataFrame(train_history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Multi-Head Feedback Model – Training Curves", fontsize=14)

for ax, (tr_col, vl_col, title) in zip(axes, [
    ("train_loss",     "val_loss",     "Total Loss"),
    ("train_span_f1",  "val_span_f1",  "Head 1: Span F1"),
    ("train_sev_rmse", "val_sev_rmse", "Head 2: Severity RMSE"),
]):
    ax.plot(df_hist["epoch"], df_hist[tr_col], label="Train", marker="o")
    ax.plot(df_hist["epoch"], df_hist[vl_col], label="Val",   marker="s", linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved as training_curves.png")

## 8. Evaluation – Per-Class Report (Head 1)

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in val_loader:
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        span_l = batch["span_labels"].to(DEVICE)
        d_ids  = batch["dec_input_ids"].to(DEVICE)
        d_mask = batch["dec_attention_mask"].to(DEVICE)
        g_lbl  = batch["gen_labels"].to(DEVICE)

        out   = model(ids, mask, d_ids, d_mask, g_lbl, span_l)
        preds = out["span_logits"].argmax(-1).view(-1).cpu().numpy()
        true  = span_l.view(-1).cpu().numpy()
        valid = true != -100
        all_preds.extend(preds[valid])
        all_true.extend(true[valid])

print("=== Head 1 – Error Span Classification Report ===")
print(classification_report(
    all_true, all_preds,
    target_names=ERROR_LABELS,
    zero_division=0
))

## 9. Confusion Matrix (Head 1)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_true, all_preds, labels=list(range(NUM_LABELS)))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=ERROR_LABELS, yticklabels=ERROR_LABELS, ax=ax
)
ax.set_title("Head 1 – Error Span Confusion Matrix")
ax.set_ylabel("True")
ax.set_xlabel("Predicted")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

## 10. Inference Pipeline

This is the structured output consumed by `Feedback_Model.py` → `Plan_Generator.py`.

In [ ]:
@torch.no_grad()
def analyze_text(
    text: str,
    cefr_level: str = "B1",
    user_history: Optional[torch.Tensor] = None,
) -> Dict:
    """
    Full inference for a single text input.

    Returns a structured dict matching the Feedback_Result model fields
    plus extra context for the Plan Generator.
    """
    model.eval()
    words = text.split()

    # Encode
    enc = enc_tokenizer(
        words,
        is_split_into_words=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    # T5 input
    prefix = f"fix CEFR {cefr_level}: {text}"
    dec_in = dec_tokenizer(
        prefix,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    # Forward (no labels → no loss computed)
    enc_out = model.encoder(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
    )
    seq_repr = enc_out.last_hidden_state
    cls_repr = seq_repr[:, 0, :]

    span_logits  = model.span_head(seq_repr)          # (1, T, 5)
    severity_val = model.severity_head(cls_repr).item()
    updated_hist, pattern_logits = model.pattern_head(cls_repr, user_history)

    # Head 3: generate feedback text
    gen_ids = model.generate_feedback(
        dec_in.input_ids, dec_in.attention_mask
    )
    corrected = dec_tokenizer.decode(gen_ids[0], skip_special_tokens=True)

    # Align span predictions back to words
    word_ids   = enc.word_ids(batch_index=0) if hasattr(enc, "word_ids") else []
    span_pred  = span_logits.argmax(-1).squeeze(0).cpu().numpy()
    word_error_map = {}  # word_idx → label
    for tok_i, wid in enumerate(enc_tokenizer(
        words, is_split_into_words=True
    ).word_ids()):
        if wid is not None and wid not in word_error_map:
            lbl = ID2LABEL.get(int(span_pred[tok_i]), "none")
            if lbl != "none":
                word_error_map[wid] = lbl

    detected_issues = [
        {"word": words[i], "position": i, "error_type": lbl}
        for i, lbl in sorted(word_error_map.items())
    ]

    # Compute per-category scores (1 – frequency of that error type)
    error_counts = Counter(d["error_type"] for d in detected_issues)
    total_words  = max(len(words), 1)

    def score_for(err_type):
        return round(1.0 - error_counts.get(err_type, 0) / total_words, 3)

    # Pattern: recurring issues
    pattern_probs  = torch.softmax(pattern_logits, dim=-1).squeeze(0).cpu().numpy()
    recurring      = [
        {"error_type": ERROR_LABELS[i], "recurrence_score": round(float(p), 3)}
        for i, p in enumerate(pattern_probs)
        if ERROR_LABELS[i] != "none" and float(p) > 0.15
    ]

    overall_score = round(1.0 - severity_val, 3)

    return {
        # ── Matches Feedback_Result model fields ──────────────────────
        "text":           text,
        "corrected":      corrected,
        "overall_score":  overall_score,
        "grammar_score":  score_for("grammar"),
        "vocab_score":    score_for("vocabulary"),
        "punct_score":    score_for("punctuation"),
        "detected_issues": [d["error_type"] for d in detected_issues],
        # ── Extra context for Plan Generator ─────────────────────────
        "severity":       round(severity_val, 4),
        "error_details":  detected_issues,
        "recurring_patterns": recurring,
        "updated_history": updated_hist,    # store per-user in DB
        "cefr_level":     cefr_level,
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
TEST_SENTENCES = [
    ("She go to school every day and she dont like homework .",          "A1"),
    ("Despite of his efforts the project fail to meet the deadline",     "B2"),
    ("The datas we collected shows a clear upward trend in engagement .","C1"),
]

for txt, lvl in TEST_SENTENCES:
    result = analyze_text(txt, cefr_level=lvl)
    print(f"\n{'='*65}")
    print(f"INPUT   : {txt}")
    print(f"LEVEL   : {lvl}")
    print(f"CORRECTED : {result['corrected']}")
    print(f"OVERALL   : {result['overall_score']}  | SEVERITY: {result['severity']}")
    print(f"GRAMMAR   : {result['grammar_score']}  | VOCAB: {result['vocab_score']}  | PUNCT: {result['punct_score']}")
    print(f"DETECTED  : {result['detected_issues']}")
    print(f"RECURRING : {result['recurring_patterns']}")

## 11. Plan-Generator Integration

Shows how the structured feedback output maps to `Plan_Generator` inputs (focus skills, level hints).

In [ ]:
def build_plan_generator_payload(
    user_id: str,
    feedback_results: List[Dict],
    mode: str = "weekly",
) -> Dict:
    """
    Aggregate multiple feedback results into the payload
    expected by Plan_Generator / Plan_Adjuster.

    Parameters
    ----------
    user_id          : str
    feedback_results : list of dicts returned by analyze_text()
    mode             : 'weekly' | 'monthly'

    Returns
    -------
    dict  – ready to POST to the GraphQL Plan mutation
    """
    if not feedback_results:
        return {}

    # Aggregate scores
    avg_overall  = round(np.mean([r["overall_score"]  for r in feedback_results]), 3)
    avg_grammar  = round(np.mean([r["grammar_score"]  for r in feedback_results]), 3)
    avg_vocab    = round(np.mean([r["vocab_score"]    for r in feedback_results]), 3)
    avg_punct    = round(np.mean([r["punct_score"]    for r in feedback_results]), 3)
    avg_severity = round(np.mean([r["severity"]       for r in feedback_results]), 3)

    # Determine focus skills from lowest scores
    score_map = {
        "grammar":     avg_grammar,
        "vocabulary":  avg_vocab,
        "punctuation": avg_punct,
    }
    focus_skills = sorted(score_map, key=score_map.get)[:2]  # worst 2

    # Collect all recurring patterns across sessions
    pattern_counter = Counter()
    for r in feedback_results:
        for p in r.get("recurring_patterns", []):
            pattern_counter[p["error_type"]] += 1

    # CEFR level: use the one from the latest result
    cefr = feedback_results[-1].get("cefr_level", "B1")

    # Build user-performance summary for the LLM prompt
    performance_summary = (
        f"User {user_id} | CEFR {cefr} | "
        f"Overall {avg_overall} | Grammar {avg_grammar} | "
        f"Vocab {avg_vocab} | Punct {avg_punct} | "
        f"Focus: {focus_skills} | "
        f"Recurring: {dict(pattern_counter.most_common(3))}"
    )

    return {
        # Direct fields for Plan_Generator mutation
        "user_id":       user_id,
        "cefr_level":    cefr,
        "mode":          mode,
        "focus_skills":  focus_skills,
        # Aggregated scores (stored in Personalized_Plan or passed as context)
        "scores": {
            "overall":     avg_overall,
            "grammar":     avg_grammar,
            "vocabulary":  avg_vocab,
            "punctuation": avg_punct,
            "severity":    avg_severity,
        },
        "recurring_issues": dict(pattern_counter.most_common(5)),
        # Human-readable prompt context for the LLM inside Plan_Generator
        "performance_summary": performance_summary,
    }


# ── Demo: simulate 3 submissions from one user ─────────────────────────────
USER_ID   = "user_demo_001"
RESULTS   = [analyze_text(txt, lvl) for txt, lvl in TEST_SENTENCES]
PAYLOAD   = build_plan_generator_payload(USER_ID, RESULTS, mode="weekly")

print(json.dumps(
    {k: v for k, v in PAYLOAD.items() if k != "performance_summary"},
    indent=2
))
print("\nPerformance summary for LLM prompt:")
print(PAYLOAD["performance_summary"])

## 12. Pattern Aggregator Visualisation (Head 4)

In [ ]:
pattern_data = []
for r in RESULTS:
    for p in r.get("recurring_patterns", []):
        pattern_data.append(p)

if pattern_data:
    df_pat = pd.DataFrame(pattern_data)
    df_agg = df_pat.groupby("error_type")["recurrence_score"].mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ["#e74c3c" if v > 0.3 else "#f39c12" if v > 0.2 else "#27ae60"
              for v in df_agg.values]
    df_agg.plot.bar(ax=ax, color=colors, edgecolor="white")
    ax.set_title(f"Head 4 – Recurring Error Patterns for {USER_ID}")
    ax.set_xlabel("Error Type")
    ax.set_ylabel("Avg Recurrence Score")
    ax.set_ylim(0, 1)
    ax.axhline(0.3, color="red",    linestyle="--", alpha=0.5, label="High (>0.3)")
    ax.axhline(0.2, color="orange", linestyle="--", alpha=0.5, label="Medium (>0.2)")
    ax.legend(fontsize=8)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig("pattern_aggregator.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("No recurring patterns detected in demo data.")

## 13. Save / Load Model

In [ ]:
import os

SAVE_DIR = "multi_head_feedback_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save encoder + heads
model.encoder.save_pretrained(os.path.join(SAVE_DIR, "encoder"))
enc_tokenizer.save_pretrained(os.path.join(SAVE_DIR, "encoder"))

# Save T5
model.t5.save_pretrained(os.path.join(SAVE_DIR, "t5"))
dec_tokenizer.save_pretrained(os.path.join(SAVE_DIR, "t5"))

# Save custom head weights + pattern history
torch.save({
    "span_head":     model.span_head.state_dict(),
    "severity_head": model.severity_head.state_dict(),
    "pattern_head":  model.pattern_head.state_dict(),
    "history_store": history_store,
    "label2id":      LABEL2ID,
    "id2label":      ID2LABEL,
}, os.path.join(SAVE_DIR, "heads.pt"))

print(f"Model saved to '{SAVE_DIR}/'")
print(os.listdir(SAVE_DIR))

In [ ]:
# ── Reload ────────────────────────────────────────────────────────────────────
loaded_model = MultiHeadFeedbackModel(
    encoder_name = os.path.join(SAVE_DIR, "encoder"),
    t5_name      = os.path.join(SAVE_DIR, "t5"),
    num_labels   = NUM_LABELS,
).to(DEVICE)

ckpt = torch.load(os.path.join(SAVE_DIR, "heads.pt"), map_location=DEVICE)
loaded_model.span_head.load_state_dict(ckpt["span_head"])
loaded_model.severity_head.load_state_dict(ckpt["severity_head"])
loaded_model.pattern_head.load_state_dict(ckpt["pattern_head"])
history_store.update(ckpt["history_store"])

print("Model reloaded successfully.")

## 14. Integration Notes

### Wiring into `Feedback_Model.py`

```python
# In services/Feedback_Model.py  ─ replace the existing Analyze_Text method
class FeedbackAnalyzer:
    def __init__(self):
        self.model = MultiHeadFeedbackModel(...)
        # load heads.pt, encoder, t5 from saved dir

    def Analyze_Text(self, Text: str, CEFR_Level="B1",
                     User_History=None) -> dict:
        return analyze_text(Text, cefr_level=CEFR_Level,
                            user_history=User_History)
```

### Wiring into `Plan_Generator.py`

```python
# After fetching all Task_Details for a user:
feedback_results = [
    analyzer.Analyze_Text(task.Input_Text, CEFR_Level=plan.Level)
    for task in recent_tasks
]
payload = build_plan_generator_payload(user_id, feedback_results, mode="weekly")

# Then pass payload["performance_summary"] as extra context to the LLM prompt
# and payload["focus_skills"] as the `Focus_Skills` argument.
```

### Storing `updated_history` per user

```python
# In MongoDB (via djongo) add a field to Classification_Level or Task_Details:
User_Pattern_History = models.JSONField(null=True, blank=True)
# Store: json.dumps(result["updated_history"].tolist())
# Load : torch.tensor(json.loads(db_obj.User_Pattern_History))
```

---

### Data Flow Summary

```
User submits text (Task_Details.Input_Text)
         │
         ▼
MultiHeadFeedbackModel.analyze_text()
   ├─ Head 1 → error spans + types  ──────────────────────┐
   ├─ Head 2 → severity score                              │
   ├─ Head 3 → corrected text + feedback sentence          ├─► Feedback_Result (MongoDB)
   └─ Head 4 → recurring pattern scores + updated history  │
                                                           │
         ▼                                                 │
build_plan_generator_payload()  ◄──────────────────────────┘
   └─ focus_skills, scores, recurring_issues
         │
         ▼
Plan_Generator (LLM) → Personalized_Plan (MongoDB)
         │
         ▼
Plan_Adjuster (on user instruction) → updated plan
```